# Sparse autoencoder

In [ ]:
import torch
from torch import nn
from torch.utils import data
import matplotlib.pyplot as plt
from config import (
    SEED,
    TRAIN_BS,
    VAL_BS,
    COLORED_PROPORTION,
    CLS_EPOCHS,
    SAE_FEATURES_LAYER,
    SAE_BS,
    SAE_HIDDEN_DIM,
    SAE_ALPHA,
    SAE_TOPK
)
from models import ClsModel, SAE, TrimmedClsModel
import train_utils
import plot_utils
import metrics
import os
import tqdm
import data_utils
import functools

In [ ]:
plt.style.use('default')

device = "cuda" if torch.cuda.is_available() else "mps"

In [ ]:
train_dataset = torch.load("./data/train_dataset.pt", weights_only=False)
val_dataset = torch.load("./data/val_dataset.pt", weights_only=False)
clean_val_dataset = torch.load("./data/clean_val_dataset.pt", weights_only=False)
fully_colored_val_dataset = torch.load("./data/fully_colored_val_dataset.pt", weights_only=False)
colors_flipped_val_dataset = torch.load("./data/colors_flipped_val_dataset.pt", weights_only=False)

train_dataloader = data.DataLoader(train_dataset, batch_size=TRAIN_BS, shuffle=True)
val_dataloader = data.DataLoader(val_dataset, batch_size=VAL_BS)
clean_val_dataloader = data.DataLoader(clean_val_dataset, batch_size=VAL_BS)
fully_colored_val_dataloader = data.DataLoader(fully_colored_val_dataset, batch_size=VAL_BS)
colors_flipped_val_dataloader = data.DataLoader(colors_flipped_val_dataset, batch_size=VAL_BS)

In [ ]:
model = ClsModel()

model.load_state_dict(torch.load(f"./weights/cls_model_{COLORED_PROPORTION}.pt"))
model.to(device)

In [ ]:
sae_input_shape = model.fc[SAE_FEATURES_LAYER].in_features

In [ ]:
sae_train_dataset = data_utils.generate_sae_dataset(model, train_dataset, device=device)
sae_val_dataset = data_utils.generate_sae_dataset(model, val_dataset, device=device)

sae_train_dataloader = data.DataLoader(sae_train_dataset, SAE_BS, shuffle=True)
sae_val_dataloader = data.DataLoader(sae_val_dataset, SAE_BS, shuffle=False)

In [ ]:
def sae_criterion_fn(outputs, targets, hidden, alpha=0.5):
    """
    SAE criterion that applies both L1 loss on hidden SAE vector as well as L2 reconstruction loss.
    
    Args:
		outputs: Output of the SAE decoder.
        targets: Target used for reconstruction loss.
        hidden: Sparse hidden vector, output of the SAE's encoder.
        alpha (optional): Weight applied to sparsity loss.
    Returns:
		torch.Tensor: Combined sparsity and reconstruction loss.
    """
    mae = torch.mean(torch.abs(hidden))	# L1 Loss to ensure hidden layer sparsity
    reconstruction_loss = torch.mean((outputs-targets)**2)
    return reconstruction_loss+alpha*mae

In [ ]:
# sae_model = TopkSAE(sae_input_shape, SAE_HIDDEN_DIM, topk=SAE_TOPK)
sae_model = SAE(sae_input_shape, SAE_HIDDEN_DIM)
sae_model.to(device)
sae_optimizer = torch.optim.Adam(sae_model.parameters(), lr=1e-3)
sae_criterion = functools.partial(sae_criterion_fn, alpha=SAE_ALPHA)
# sae_criterion = nn.MSELoss()

In [ ]:
train_utils.sae_train(sae_model, sae_train_dataloader, sae_val_dataloader, sae_criterion, sae_optimizer, epochs=30, device=device)

In [ ]:
plot_utils.show_from_dataset(val_dataset, range(0, 10))
plot_utils.show_from_dataset(colors_flipped_val_dataset, range(0, 10))

In [ ]:
for i in range(10):
	input, target = val_dataset[i]
	input = input.to(device)
	input = input.unsqueeze(0)
	embedding = model.fc[:SAE_FEATURES_LAYER](input)

	output, hidden = sae_model(embedding)
	values, indices = torch.topk(hidden, SAE_TOPK)
	print(f"Colors not flipped {i}: {indices}")

	input, target = colors_flipped_val_dataset[i]
	input = input.to(device)
	input = input.unsqueeze(0)
	embedding = model.fc[:SAE_FEATURES_LAYER](input)

	output, hidden = sae_model(embedding)
	values, indices = torch.topk(hidden, SAE_TOPK)
	print(f"Colors flipped {i}: {indices}")

In [ ]:
@torch.no_grad()
def get_representatives(cls_model, sae_model, dataloader, n_representatives=10):
	"""
	Given classification model and sae model returns n_representatives examples from dataset which activate each SAE hidden layer neurons the most
	"""
	sae_hidden_dim = sae_model.hidden_dim

	top_values = torch.full((sae_hidden_dim, n_representatives), -float('inf'), device=device)

	representatives = torch.full((sae_hidden_dim, n_representatives), -1, dtype=torch.long, device=device)

	current_img_offset = 0

	for inputs, targets in tqdm.tqdm(dataloader):
		inputs = inputs.to(device)
		bs = inputs.size(0)
		
		embeddings = cls_model.fc[:SAE_FEATURES_LAYER](inputs)
		
		output, hidden = sae_model(embeddings)
		
		batch_indices = torch.arange(current_img_offset, current_img_offset + bs, device=device)
		
		batch_hidden = hidden.T 
		
		# Create dataset indices for the current batch
		batch_indices = batch_indices.unsqueeze(0).expand(sae_hidden_dim, bs)
		
		# Concatenate the running top N with the new batch
		combined_values = torch.cat([top_values, batch_hidden], dim=1)
		combined_indices = torch.cat([representatives, batch_indices], dim=1)
		
		# Find the global winners so far
		top_values, topk_args = torch.topk(combined_values, k=n_representatives, dim=1)
		
		# Gather the winning image indices
		representatives = torch.gather(combined_indices, dim=1, index=topk_args)
		
		current_img_offset += bs

	return representatives, top_values

In [ ]:
n_representatives = 10

clean_val_representatives, clean_val_top_values = get_representatives(model, sae_model, clean_val_dataloader, n_representatives)
fully_colored_val_representatives, fully_colored_val_top_values = get_representatives(model, sae_model, fully_colored_val_dataloader, n_representatives)
colors_flipped_val_representatives, colors_flipped_val_top_values = get_representatives(model, sae_model, colors_flipped_val_dataloader, n_representatives)

In [ ]:
neurons_to_show = min(5, SAE_HIDDEN_DIM)
print(f"Showing representative samples for {neurons_to_show} out of {SAE_HIDDEN_DIM} sae neurons")

for neuron_idx in range(neurons_to_show):
    print(f"Neuron: {neuron_idx}")
    plot_utils.show_from_dataset(clean_val_dataset, clean_val_representatives[neuron_idx].tolist())
    plot_utils.show_from_dataset(fully_colored_val_dataset, fully_colored_val_representatives[neuron_idx].tolist())
    plot_utils.show_from_dataset(colors_flipped_val_dataset, colors_flipped_val_representatives[neuron_idx].tolist())

# Check representatives for features activating at specific dataset examples

In [ ]:
example_idx = 2
top_k = 5
experiment_dataset = clean_val_dataset  # Change this to colors_flipped_val_dataset to test on the colors flipped dataset

print(f"Label: {experiment_dataset[example_idx][1]}")

inputs = experiment_dataset[example_idx][0].unsqueeze(0).to(device)
sae_input = model.fc[:SAE_FEATURES_LAYER](inputs)

sae_output, sae_hidden = sae_model(sae_input)
topk_features = torch.topk(sae_hidden, top_k).indices.squeeze().tolist()

plt.imshow(inputs.squeeze().cpu().permute(1, 2, 0))
plt.axis('off')
plt.title(f"Example {example_idx} from validation dataset")
plt.show()

print(f"TOP {top_k} features for example {example_idx}: {topk_features}")

for i in range(top_k):
    print("\n==============")
    print(f"Representative samples for feature {topk_features[i]}:")
    print(f"Values: {clean_val_top_values[topk_features[i]].tolist()}")
    plot_utils.show_from_dataset(clean_val_dataset, clean_val_representatives[topk_features[i]].tolist())
    print(f"Values: {fully_colored_val_top_values[topk_features[i]].tolist()}")
    plot_utils.show_from_dataset(fully_colored_val_dataset, fully_colored_val_representatives[topk_features[i]].tolist())
    print(f"Values: {colors_flipped_val_top_values[topk_features[i]].tolist()}")
    plot_utils.show_from_dataset(colors_flipped_val_dataset, colors_flipped_val_representatives[topk_features[i]].tolist())

# Validate model with SAE neurons killed that ativate for certain colors

In [ ]:
neurons_to_kill = [2, 4, 5, 6, 7, 14, 18, 31, 38, 42, 44, 47 ]

In [ ]:
trimmed_model = TrimmedClsModel(model, sae_model, neurons_to_kill)

cls_criterion = nn.CrossEntropyLoss()

metrics_calculator = metrics.MetricsCalculator()

In [ ]:
# Evaluate the model on clean validation dataset
clean_val_loss = train_utils.cls_evaluate(trimmed_model, clean_val_dataloader, cls_criterion, metrics_calculator, device)
clean_val_accuracy, clean_val_precision, clean_val_recall, clean_val_f1_score, clean_val_auprc, clean_val_auroc = metrics_calculator.compute_all()

# Evaluate the model on fully colored validation dataset
fully_colored_val_loss = train_utils.cls_evaluate(trimmed_model, fully_colored_val_dataloader, cls_criterion, metrics_calculator, device)
fully_colored_val_accuracy, fully_colored_val_precision, fully_colored_val_recall, fully_colored_val_f1_score, fully_colored_val_auprc, fully_colored_val_auroc = metrics_calculator.compute_all()

colors_flipped_val_loss = train_utils.cls_evaluate(trimmed_model, colors_flipped_val_dataloader, cls_criterion, metrics_calculator, device)
colors_flipped_val_accuracy, colors_flipped_val_precision, colors_flipped_val_recall, colors_flipped_val_f1_score, colors_flipped_val_auprc, colors_flipped_val_auroc = metrics_calculator.compute_all()

print("Performance on clean validation set:")
print(f"Clean Val Loss: {clean_val_loss:.4f}, Accuracy: {clean_val_accuracy:.4f}, Precision: {clean_val_precision:.4f}, Recall: {clean_val_recall:.4f}, F1 Score: {clean_val_f1_score:.4f}, AUPRC: {clean_val_auprc:.4f}, AUROC: {clean_val_auroc:.4f}")
print("Performance on fully colored validation set:")
print(f"Fully Colored Val Loss: {fully_colored_val_loss:.4f}, Accuracy: {fully_colored_val_accuracy:.4f}, Precision: {fully_colored_val_precision:.4f}, Recall: {fully_colored_val_recall:.4f}, F1 Score: {fully_colored_val_f1_score:.4f}, AUPRC: {fully_colored_val_auprc:.4f}, AUROC: {fully_colored_val_auroc:.4f}")

print("Performance on validation set with colors flipped:")
print(f"Colors flipped Val Loss: {colors_flipped_val_loss:.4f}, Accuracy: {colors_flipped_val_accuracy:.4f}, Precision: {colors_flipped_val_precision:.4f}, Recall: {colors_flipped_val_recall:.4f}, F1 Score: {colors_flipped_val_f1_score:.4f}, AUPRC: {colors_flipped_val_auprc:.4f}, AUROC: {colors_flipped_val_auroc:.4f}")
